# **Modelo Supervisado de Clasificación**


Este notebook implementa el modelo supervisado de clasificación para predecir la severidad
de los accidentes de tránsito registrados en México durante 2024, utilizando el dataset
limpio generado en el notebook de análisis exploratorio y tomando en cuenta las observaciones 
hechas en en notebook de análisis descriptivo estadístico.

La variable objetivo es `CLASACC`, que distingue tres niveles de severidad:
- **Sólo daños:** el accidente únicamente generó daños materiales (~82% de los registros)
- **No fatal:** el accidente dejó personas heridas pero sin fallecimientos (~17%)
- **Fatal:** el accidente resultó en al menos una persona fallecida (~1.1%)

El desbalance severo entre clases es el principal desafío técnico de esta tarea y guiará
las decisiones de preprocesamiento, selección de modelo y evaluación.

# **Estrategia General**

Antes de comenzar con la carga de datos y el preprocesamiento, se define la estrategia
general de modelado que guiará todas las decisiones técnicas de este notebook.

---

## **Tarea y variable objetivo**

La tarea es una **clasificación multiclase** sobre la variable `CLASACC`, que distingue
tres niveles de severidad de los accidentes de tránsito:

- `Sólo daños` — únicamente daños materiales (~82% de los registros)
- `No fatal` — personas heridas sin fallecimientos (~17%)
- `Fatal` — al menos una persona fallecida (~1.1%)

El **desbalance severo** entre clases es el principal desafío técnico. Un modelo que ignore
completamente la clase `Fatal` puede alcanzar 98% de accuracy y ser completamente inútil
para el objetivo del proyecto. Por esta razón, la métrica principal de comparación será el
**F1-score macro**, que pondera por igual las tres clases sin importar su frecuencia.

---

## **Modelos**

Se implementan cuatro modelos en orden creciente de complejidad:

**Línea base trivial** — Clasificador mayoritario que siempre predice `Sólo daños`.
Alcanzará ~82% de accuracy con 0% de recall en `Fatal` y `No fatal`. Es el piso mínimo
obligatorio que cualquier modelo real debe superar.

**Logistic Regression** — Modelo lineal interpretable y rápido. Funciona como segunda
línea de referencia antes de modelos más complejos. Maneja el desbalance de clases
mediante `class_weight='balanced'`.

**Random Forest** — Modelo principal del proyecto. Maneja variables mixtas, es robusto
frente al desbalance con `class_weight='balanced'`, escala bien con 374,078 registros y
produce importancia de características de forma nativa, insumo directo para la etapa
de agrupamiento no supervisado del proyecto.

**XGBoost** — Modelo de reserva. Se entrenará únicamente si Random Forest no produce
resultados satisfactorios. Generalmente supera a Random Forest en datasets tabulares con
desbalance severo, pero requiere mayor costo computacional y ajuste de hiperparámetros.

---

## **Plan de contingencia**

Si los resultados de la clasificación multiclase son insatisfactorios, particularmente
si el recall de `Fatal` es inaceptablemente bajo en todos los modelos, se reformulará
el problema como **clasificación binaria**: `Fatal` vs `No fatal + Sólo daños`, poniendo
el foco en identificar correctamente los accidentes más graves.

---

### **Configuración Global**

Las siguientes constantes controlan el comportamiento reproducible del notebook.
`RANDOM_STATE` se usa como semilla en todos los procesos estocásticos: división
de datos, entrenamiento de modelos y ajuste de hiperparámetros.

In [1]:
RANDOM_STATE = 42

## **1. Carga y Verificación del Dataset**

En esta sección se carga el dataset limpio generado en el notebook de análisis exploratorio
(`atus_anual_2024_limpio.csv`) utilizando la clase `DataRepository`, que abstrae el acceso
al archivo y valida automáticamente que el dataset contenga las columnas mínimas requeridas
antes de continuar con el pipeline de modelado.

In [2]:
import sys
sys.path.append('../src/')

from data_repository import DataRepository

repo = DataRepository("../data/atus_anual_2024_limpio.csv")
df = repo.load()

print(repo.summary())

           RESUMEN DEL DATASET CARGADO
  Ruta:          ../data/atus_anual_2024_limpio.csv
  Configuración: /home/ofgm/Documentos/Tareas_Almacenes-y-Mineria-de-Datos/ProyectoFinal/notebooks/../src/config/repository_config.yaml
  Filas:         374,078
  Columnas:      42

  Distribución de CLASACC (variable objetivo):
    Sólo daños   306,088  (81.82%)
    No fatal      63,809  (17.06%)
    Fatal          4,181  (1.12%)

  Valores nulos por columna:
    ID_EDAD                    96,222


El resumen confirma que el dataset fue cargado correctamente y que la distribución de
`CLASACC` presenta el desbalance severo esperado: aproximadamente 82% de los registros
corresponden a `Sólo daños`, 17% a `No fatal` y apenas 1.1% a `Fatal`. Este desbalance
es el principal desafío técnico del modelo y guiará las decisiones de preprocesamiento
y evaluación en las secciones siguientes.

## **2. Selección y Justificación de Variables de Entrada**

Antes de construir el pipeline de preprocesamiento, se analizan todas las variables disponibles
en el dataset limpio para determinar cuáles se incluyen como features del modelo y cuáles se
excluyen. Las decisiones se toman con base en el EDA, el dominio del problema y criterios
técnicos de modelado.

---

### Variables excluidas

| Variable | Razón |
|---|---|
| `ID_MINUTO` | Granularidad excesiva sin relación causal con la severidad del accidente |
| `ID_DIA` | El día del mes no tiene relación causal con la severidad |
| `NOM_MUN` | 1,435 categorías únicas — cardinalidad demasiado alta para encoding |
| `TRANVIA` | Prácticamente sin variación real (2 valores únicos) |
| `FERROCARRI` | Prácticamente sin variación real (2 valores únicos) |
| `OTROVEHIC` | Prácticamente sin variación real (2 valores únicos) |
| `TOTAL_MUERTOS` | **Data leakage** — variable derivada directamente de la objetivo |
| `CONDMUERTO`, `CONDHERIDO` | **Data leakage** — el número de víctimas es consecuencia, no causa |
| `PASAMUERTO`, `PASAHERIDO` | **Data leakage** — ídem |
| `PEATMUERTO`, `PEATHERIDO` | **Data leakage** — ídem |
| `CICLMUERTO`, `CICLHERIDO` | **Data leakage** — ídem |
| `OTROMUERTO`, `OTROHERIDO` | **Data leakage** — ídem |

Las columnas de víctimas merecen una explicación especial. Si bien tienen una correlación
directa con `CLASACC`, un accidente con `CONDMUERTO > 0` es casi inevitablemente `Fatal`,
**incluirlas produciría un modelo trivialmente perfecto pero inútil en producción**. En un
escenario real, la severidad del accidente se desconoce en el momento en que se quiere
predecir; las víctimas son una consecuencia del accidente, no una característica observable
previa a su clasificación.

---

### Variables numéricas

| Variable | Transformación |
|---|---|
| `ID_EDAD` | Imputación con mediana (solo entrenamiento) + StandardScaler |
| `AUTOMOVIL`, `CAMPASAJ`, `MICROBUS`, `PASCAMION` | StandardScaler |
| `OMNIBUS`, `CAMIONETA`, `CAMION`, `TRACTOR` | StandardScaler |
| `MOTOCICLET`, `BICICLETA` | StandardScaler |
| `CONDUCTOR_FUGADO`, `EDAD_DESCONOCIDA` | Binarias — sin escalado |

---

### Variables temporales — Encoding cíclico

`MES` e `ID_HORA` tienen naturaleza circular: la hora 23 está temporalmente cerca de la
hora 0, y diciembre está cerca de enero. Un escalado lineal no captura esta continuidad.
Por esta razón se aplica **encoding cíclico** mediante transformaciones seno/coseno:

$$\text{MES\_SEN} = \sin\left(\frac{2\pi \cdot \text{MES}}{12}\right) \quad
\text{MES\_COS} = \cos\left(\frac{2\pi \cdot \text{MES}}{12}\right)$$

$$\text{HORA\_SEN} = \sin\left(\frac{2\pi \cdot \text{ID\_HORA}}{24}\right) \quad
\text{HORA\_COS} = \cos\left(\frac{2\pi \cdot \text{ID\_HORA}}{24}\right)$$

Las columnas originales `MES` e `ID_HORA` se eliminan tras la transformación.

---

### Variables categóricas — One-Hot Encoding

| Variable | Valores únicos |
|---|---|
| `DIASEMANA` | 7 |
| `URBANA` | 3 |
| `SUBURBANA` | 4 |
| `TIPACCID` | 12 |
| `CAUSAACCI` | 5 |
| `CAPAROD` | 2 |
| `SEXO` | 3 |
| `ALIENTO` | 3 |
| `CINTURON` | 3 |
| `NOM_ENT` | 32 |

Se aplica `OneHotEncoder` con `drop='first'` para evitar multicolinealidad perfecta entre
las categorías de cada variable.

---

### Resumen

En total se incluyen **26 variables originales** que tras las transformaciones producen un
espacio de features con codificación numérica completa, libre de data leakage y lista para
ser consumida por los modelos de clasificación.

In [3]:
from preprocessor import Preprocessor
from sklearn.model_selection import train_test_split

# Separación de features y variable objetivo
y = df[repo.target_column]
X = df.drop(columns=[repo.target_column])

print(f"Shape de X: {X.shape}")
print(f"Shape de y: {y.shape}")
print(f"\nDistribución de y:")
print(y.value_counts())

Shape de X: (374078, 41)
Shape de y: (374078,)

Distribución de y:
CLASACC
Sólo daños    306088
No fatal       63809
Fatal           4181
Name: count, dtype: int64


In [4]:
# División en entrenamiento y prueba antes de ajustar el preprocesador
# IMPORTANTE: el preprocesador se ajusta SOLO sobre X_train para
# evitar data leakage en la imputación de ID_EDAD
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y      # Preserva la distribución de clases en ambos conjuntos
)

print(f"X_train: {X_train.shape}  |  X_test: {X_test.shape}")
print(f"\nDistribución en entrenamiento:")
print(y_train.value_counts(normalize=True).mul(100).round(2))
print(f"\nDistribución en prueba:")
print(y_test.value_counts(normalize=True).mul(100).round(2))

X_train: (299262, 41)  |  X_test: (74816, 41)

Distribución en entrenamiento:
CLASACC
Sólo daños    81.82
No fatal      17.06
Fatal          1.12
Name: proportion, dtype: float64

Distribución en prueba:
CLASACC
Sólo daños    81.82
No fatal      17.06
Fatal          1.12
Name: proportion, dtype: float64


In [5]:
# Ajuste y transformación
prep = Preprocessor()
print(prep.summary())

X_train_t = prep.fit_transform(X_train)
X_test_t  = prep.transform(X_test)

print(f"\nShape X_train transformado: {X_train_t.shape}")
print(f"Shape X_test transformado:  {X_test_t.shape}")
print(f"Total features:             {len(prep.feature_names)}")

        CONFIGURACIÓN DEL PREPROCESADOR
  Configuración: /home/ofgm/Documentos/Tareas_Almacenes-y-Mineria-de-Datos/ProyectoFinal/notebooks/../src/config/preprocessor_config.yaml

  Columnas excluidas      (17):
    - ID_MINUTO
    - ID_DIA
    - NOM_MUN
    - TRANVIA
    - FERROCARRI
    - OTROVEHIC
    - TOTAL_MUERTOS
    - CONDMUERTO
    - CONDHERIDO
    - PASAMUERTO
    - PASAHERIDO
    - PEATMUERTO
    - PEATHERIDO
    - CICLMUERTO
    - CICLHERIDO
    - OTROMUERTO
    - OTROHERIDO

  Numéricas               (11): Imputer(mediana) + StandardScaler
    - ID_EDAD
    - AUTOMOVIL
    - CAMPASAJ
    - MICROBUS
    - PASCAMION
    - OMNIBUS
    - CAMIONETA
    - CAMION
    - TRACTOR
    - MOTOCICLET
    - BICICLETA

  Binarias                (2): passthrough
    - CONDUCTOR_FUGADO
    - EDAD_DESCONOCIDA

  Categóricas             (10): OneHotEncoder(drop='first')
    - DIASEMANA
    - URBANA
    - SUBURBANA
    - TIPACCID
    - CAUSAACCI
    - CAPAROD
    - SEXO
    - ALIENTO
    - CINT

### Resultado del preprocesamiento

El pipeline transforma las 26 variables originales seleccionadas en **81 features numéricas**
listas para el modelado. El incremento se debe principalmente al One-Hot Encoding de las
variables categóricas, que expande cada categoría en una columna binaria independiente.

**Sobre la división antes del preprocesamiento:**
La separación en `X_train` y `X_test` se realizó **antes** de ajustar el preprocesador,
y este se ajustó **exclusivamente sobre `X_train`**. Esto es fundamental para evitar
*data leakage* en la etapa de imputación de `ID_EDAD`.

`SimpleImputer` calcula la mediana de `ID_EDAD` durante `fit()` y la usa para rellenar
los valores `NaN` en cualquier conjunto posterior. Si se ajustara sobre el dataset completo,
la mediana incorporaría información de los datos de prueba, dándole al modelo una ventaja
irreal que no existiría en producción. Al ajustar solo sobre `X_train`, la mediana refleja
únicamente lo que el modelo "conoce" durante el entrenamiento, y esa misma mediana se aplica
a `X_test` y a cualquier dato nuevo, replicando fielmente el comportamiento en producción.

El parámetro `stratify=y` en `train_test_split` garantiza que la distribución de `CLASACC`
se preserve en ambos conjuntos, lo cual es especialmente importante dado el desbalance
severo de la clase `Fatal` (~1.1%).

## **4. Inicialización de Modelos**

Se instancian los modelos usando `ModelFactory`, que centraliza la creación y configuración
de cada algoritmo. Los hiperparámetros base se cargan desde el archivo YAML de configuración
y `RANDOM_STATE` sobreescribe la semilla en todos los modelos para garantizar
reproducibilidad.

In [6]:
from model_factory import ModelFactory

print(ModelFactory.summary())

         MODELOS DISPONIBLES — ModelFactory
  Configuración: /home/ofgm/Documentos/Tareas_Almacenes-y-Mineria-de-Datos/ProyectoFinal/notebooks/../src/config/model_factory_config.yaml

  [dummy]
    strategy                  most_frequent

  [logistic_regression]
    max_iter                  1000
    class_weight              balanced
    random_state              42
    solver                    lbfgs

  [random_forest]
    n_estimators              100
    max_depth                 None
    min_samples_split         2
    min_samples_leaf          1
    class_weight              balanced
    random_state              42
    n_jobs                    -1

  [xgboost]
    n_estimators              100
    max_depth                 6
    learning_rate             0.1
    subsample                 0.8
    colsample_bytree          0.8
    use_label_encoder         False
    eval_metric               mlogloss
    random_state              42
    n_jobs                    -1



In [7]:
dummy = ModelFactory.create('dummy')
lr    = ModelFactory.create('logistic_regression', random_state=RANDOM_STATE)
rf    = ModelFactory.create('random_forest',       random_state=RANDOM_STATE)

print("Modelos inicializados:")
print(f"  Línea base:          {dummy}")
print(f"  Logistic Regression: {lr}")
print(f"  Random Forest:       {rf}")

Modelos inicializados:
  Línea base:          DummyClassifier(strategy='most_frequent')
  Logistic Regression: LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)
  Random Forest:       RandomForestClassifier(class_weight='balanced', n_jobs=-1, random_state=42)


Los tres modelos están configurados con `class_weight='balanced'` donde aplica, lo que
le indica a scikit-learn que compense automáticamente el desbalance de clases ajustando
el peso de cada clase inversamente proporcional a su frecuencia. Esto es especialmente
importante para la clase `Fatal`, que representa apenas el 1.1% de los registros.

## 5. **Entrenamiento de Modelos**

Se entrena cada modelo usando `Trainer`, que encapsula la validación cruzada, el ajuste
de hiperparámetros con `GridSearchCV` y la medición del tiempo de entrenamiento. Los
modelos se entrenan uno a la vez para mantener la trazabilidad de cada experimento.

La métrica principal de optimización es **F1-score macro**, que pondera por igual las
tres clases de `CLASACC` independientemente de su frecuencia.

In [8]:
from trainer import Trainer

trainer = Trainer(random_state=RANDOM_STATE)

### 5.1 Línea Base Trivial — Clasificador Mayoritario

In [9]:
best_dummy, results_dummy = trainer.fit(dummy, X_train_t, y_train, 'dummy')
print(trainer.summary())
trainer.save_model(best_dummy, 'dummy', output_dir='../models')

[dummy] Ejecutando validación cruzada (3 folds)...
[dummy] CV f1_macro: 0.3000 ± 0.0000 (3.78s)
[dummy] Entrenando modelo (sin GridSearchCV)...
[dummy] Entrenamiento completado (0.19s)
   RESULTADOS DE ENTRENAMIENTO — dummy
  Validación cruzada (3 folds, f1_macro):
    Scores por fold:  [0.3 0.3 0.3]
    Media:            0.3000
    Desv. estándar:   0.0000
    Tiempo CV:        3.78s

  Ajuste de hiperparámetros (GridSearchCV):
    Sin GridSearchCV (param_grid vacío)

  Tiempo GridSearchCV:  0.19s
  Tiempo total:         3.97s
[dummy] Modelo guardado en: ../models/dummy.joblib


PosixPath('../models/dummy.joblib')

### 5.2 Logistic Regression

In [13]:
from sklearn.model_selection import train_test_split

# Muestra estratificada del 30% para Logistic Regression
X_train_lr, _, y_train_lr, _ = train_test_split(
    X_train_t, y_train,
    train_size=0.3,
    random_state=RANDOM_STATE,
    stratify=y_train
)

print(f"Shape X_train_lr: {X_train_lr.shape}")
print(f"Distribución y_train_lr:")
print(y_train_lr.value_counts(normalize=True).mul(100).round(2))

Shape X_train_lr: (89778, 81)
Distribución y_train_lr:
CLASACC
Sólo daños    81.83
No fatal      17.06
Fatal          1.12
Name: proportion, dtype: float64


In [14]:
best_lr, results_lr = trainer.fit(lr, X_train_lr, y_train_lr, 'logistic_regression')
print(trainer.summary())
trainer.save_model(best_lr, 'logistic_regression')

[logistic_regression] Ejecutando validación cruzada (3 folds)...
[logistic_regression] CV f1_macro: 0.4825 ± 0.0028 (9.47s)
[logistic_regression] Ejecutando GridSearchCV...


/home/ofgm/Documentos/Tareas_Almacenes-y-Mineria-de-Datos/ProyectoFinal/.venv/lib/python3.13/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/home/ofgm/Documentos/Tareas_Almacenes-y-Mineria-de-Datos/ProyectoFinal/.venv/lib/python3.13/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/home/ofgm/Documentos/Tareas_Almacenes-y-Mineria-de-Datos/ProyectoFinal/.venv/lib/python3.13/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/home/ofgm/Documentos/Tareas_Almacenes-y-Mineria-de-Datos/ProyectoFinal/.venv/lib/python3.13/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/home/ofgm/Documentos/Tareas_Alm

[logistic_regression] Mejores parámetros: {'C': 10.0, 'solver': 'saga'}
[logistic_regression] Mejor f1_macro: 0.4834 (523.83s)
   RESULTADOS DE ENTRENAMIENTO — logistic_regression
  Validación cruzada (3 folds, f1_macro):
    Scores por fold:  [0.4815 0.4863 0.4797]
    Media:            0.4825
    Desv. estándar:   0.0028
    Tiempo CV:        9.47s

  Ajuste de hiperparámetros (GridSearchCV):
    C                         10.0
    solver                    saga
    Mejor f1_macro:      0.4834

  Tiempo GridSearchCV:  523.83s
  Tiempo total:         533.3s
[logistic_regression] Modelo guardado en: models/logistic_regression.joblib


/home/ofgm/Documentos/Tareas_Almacenes-y-Mineria-de-Datos/ProyectoFinal/.venv/lib/python3.13/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


PosixPath('models/logistic_regression.joblib')

### 5.3 Random Forest

In [ ]:
best_rf, results_rf = trainer.fit(rf, X_train_t, y_train, 'random_forest')
print(trainer.summary())
trainer.save_model(best_rf, 'random_forest')

[random_forest] Ejecutando validación cruzada (3 folds)...
